In [9]:
import re
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

def calculate_importance_weights(candidate_sequence, reference_sequence, glossary, top_n=100):
   
    combined_texts = [candidate_sequence, reference_sequence]

    vectorizer = TfidfVectorizer(vocabulary=glossary, ngram_range=(1, 3))
    tfidf_matrix = vectorizer.fit_transform(combined_texts)
    feature_names = vectorizer.get_feature_names_out()

    tfidf_scores = tfidf_matrix.toarray().sum(axis=0)

    df_scores = pd.DataFrame({
        'phrase': feature_names,
        'score': tfidf_scores
    })

    top_phrases = df_scores.nlargest(top_n, 'score')

    scaler = MinMaxScaler()
    top_phrases['normalized_score'] = scaler.fit_transform(top_phrases[['score']])

    weights = dict(zip(top_phrases['phrase'], top_phrases['normalized_score']))
    return weights
    
def create_glossary(sentences, min_frequency=2, ngram_range=(1, 3), custom_stopwords=None):

   
    cleaned_sentences = [re.sub(r'\W+', ' ', sentence.lower()) for sentence in sentences]
  
    vectorizer = CountVectorizer(ngram_range=ngram_range, stop_words='english')
    term_matrix = vectorizer.fit_transform(cleaned_sentences)
    terms = vectorizer.get_feature_names_out()
    term_counts = term_matrix.sum(axis=0).A1  

   
    term_data = list(zip(terms, term_counts))
    glossary = [term for term, count in term_data if count >= min_frequency]

    if custom_stopwords:
        glossary = [term for term in glossary if term not in custom_stopwords]

    return glossary

from collections import Counter

def calculate_scores(original_text, predicted_text, term_weights=None):
    """
    Calculate Precision, Recall, F1-Score, and LTScore for legal terms.
    
    Parameters:
        original_text (str): Ground truth text.
        predicted_text (str): Predicted text.
        term_weights (dict): Optional; weights for specific legal terms.
    
    Returns:
        dict: Scores including Precision, Recall, F1-Score, and LTScore.
    """
    # Tokenize texts into terms (split by spaces for simplicity)
    original_terms = set(original_text.lower().split())
    predicted_terms = set(predicted_text.lower().split())

    # Compute True Positives, False Positives, and False Negatives
    true_positives = original_terms & predicted_terms
    false_positives = predicted_terms - original_terms
    false_negatives = original_terms - predicted_terms

    # Base Precision, Recall, and F1-Score
    precision = len(true_positives) / (len(true_positives) + len(false_positives)) if predicted_terms else 0
    recall = len(true_positives) / (len(true_positives) + len(false_negatives)) if original_terms else 0
    f1_score = (2 * precision * recall) / (precision + recall) if precision + recall > 0 else 0

    # Weighted LTScore if term_weights are provided
    lt_score = 0
    if term_weights:
        # Apply weights to True Positives, False Positives, and False Negatives
        tp_weighted = sum(term_weights.get(term, 1) for term in true_positives)
        fp_weighted = sum(term_weights.get(term, 1) for term in false_positives)
        fn_weighted = sum(term_weights.get(term, 1) for term in false_negatives)

        precision_weighted = tp_weighted / (tp_weighted + fp_weighted) if tp_weighted + fp_weighted > 0 else 0
        recall_weighted = tp_weighted / (tp_weighted + fn_weighted) if tp_weighted + fn_weighted > 0 else 0
        lt_score = (2 * precision_weighted * recall_weighted) / (precision_weighted + recall_weighted) if precision_weighted + recall_weighted > 0 else 0

    return {
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1_score,
        "LTScore": lt_score if term_weights else "N/A"
    }



In [4]:
import pandas as pd

df = pd.read_csv('output.csv')

In [5]:
df

,Summary,LEDSummary
0,Section 86(1)(f) vests a statutory jurisdictio...,In the case of Gujarat Urja Vikas Nigam Limite...
1,The petitioner apprehended arrest under Sectio...,The case was taken up out of turn on the basis...
2,The petitioner was arrested under Sections 344...,The petitioner is running Dance Party from man...
3,In matters concerning administrative appointme...,The petitioners were appointed as Inspectors i...
4,The facts and information of the suspected off...,The FIR was registered on the basis of a repor...
5,"In the present strained circumstances, it is n...",The petition was filed by Sunil Dalal Sr. Adv....
6,On deciding a case regarding the contradiction...,It was contended that the advertisement prescr...
7,In the case of Parneet Kaur Vs State of Punjab...,The case was registered under section 364/ 302...
8,"Section 73 of the railway’s act, 1989 provides...",The appeal was filed under Section 23 of the R...
9,It is well settled law that this Court does no...,"In the instant case, the death of deceased Bab..."


In [6]:
# Example usage
sentences =df["Summary"].values.tolist()
LEDSummary =df["LEDSummary"].values.tolist()
custom_stopwords = ["court", "law"]
glossary = create_glossary(sentences, min_frequency=2, ngram_range=(1, 3), custom_stopwords=custom_stopwords)
print(glossary)


['000', '10', '11', '11 1996', '11 1996 act', '11 babulal', '13', '147', '147 302', '147 302 149', '149', '149 323', '149 323 149', '149 324', '149 324 149', '149 ipc', '1987', '1989', '1995', '1996', '1996 act', '1997', '1998', '2003', '2003 act', '2010', '2011', '2016', '2019', '2020', '2021', '2021 case', '2021 mr', '21', '23', '274', '274 2010', '30', '302', '302 149', '309', '309 constitution', '323', '323 149', '324', '324 149', '324 149 323', '325', '325 149', '325 149 324', '326', '493', '493 2010', '73', '73 railway', '73 railway act', '86', '86 2003', '86 vests', '86 vests statutory', 'abetment', 'according', 'according fir', 'accused', 'accused assaulting', 'accused assaulting mother', 'accused petitioners', 'act', 'act 1987', 'act 1989', 'act section', 'additional', 'additional public', 'additional public prosecutor', 'adjudicate', 'adjudicate disputes', 'adjudicate disputes licensees', 'administration', 'administration reweigh', 'administration reweigh consignment', 'adver

In [12]:
sentences =df["Summary"].values.tolist()
LEDSummary =df["LEDSummary"].values.tolist()
precision=[]
recall=[]
f1score=[]
glossary = glossary
for i in range(len(sentences)):
    candidate_sequence = sentences[i]
    reference_sequence = LEDSummary[i]
    importance_weights = calculate_importance_weights(candidate_sequence, reference_sequence, glossary)
    scores = calculate_scores(candidate_sequence, reference_sequence, importance_weights)
    precision.append(scores["Precision"])
    recall.append(scores["Recall"])
    f1score.append(scores["F1-Score"])


In [16]:
finalprecision=sum(precision)/len(precision)
finalrecall=sum(recall)/len(recall)
finalf1scoren=sum(f1score)/len(f1score)

In [17]:
print(f"The LTSScorePrecision : {finalprecision}")
print(f"The LTSScoreRecall : {finalrecall}")
print(f"The LTSScoreF1Score : {finalf1scoren}")

The LTSScorePrecision : 0.47449112601884186
The LTSScoreRecall : 0.3891571842866688
The LTSScoreF1Score : 0.4219560860211038
